# Train PhoBERT Intent Classification
Fine-tune PhoBERT cho Task 2.3 - Intent Classification

In [1]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
!pip install transformers datasets scikit-learn -q

In [3]:
import json, os, random, copy
import numpy as np
from collections import Counter

DATA_PATH  = '/content/drive/MyDrive/BTL_NLP/data/annotated_intent.json'
MODEL_SAVE = '/content/drive/MyDrive/BTL_NLP/models/intent_phobert'
os.makedirs(MODEL_SAVE, exist_ok=True)

with open(DATA_PATH, encoding='utf-8') as f:
    raw = json.load(f)

texts  = [d['clause'] for d in raw]
labels = [d['intent'] for d in raw]

INTENT_LABELS = ['Obligation', 'Prohibition', 'Right', 'Termination Condition']
LABEL2ID = {l: i for i, l in enumerate(INTENT_LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}

print(f'Loaded {len(texts)} samples')
counts = Counter(labels)
for lbl in INTENT_LABELS:
    print(f'  {lbl:25}: {counts[lbl]}')

In [4]:
SYNONYMS = {
    'phải': ['có nghĩa vụ', 'có trách nhiệm', 'bắt buộc phải', 'cần phải'],
    'có quyền': ['được phép', 'được quyền', 'có thể'],
    'không được': ['không được phép', 'cấm', 'không có quyền'],
    'chấm dứt': ['kết thúc', 'hết hiệu lực', 'chấm dứt hiệu lực'],
    'hợp đồng': ['hợp đồng này', 'thỏa thuận này'],
    'Bên A': ['Bên sử dụng lao động', 'Bên thuê'],
    'Bên B': ['Bên lao động', 'Bên được thuê'],
    'thanh toán': ['chi trả', 'trả'],
    'bồi thường': ['đền bù', 'bồi hoàn'],
    'vi phạm': ['không tuân thủ', 'không thực hiện đúng'],
    'thông báo': ['báo trước', 'gửi thông báo'],
}

def synonym_replace(text, n_replacements=1):
    new_text = text
    keys = [k for k in SYNONYMS if k in text]
    if not keys:
        return text
    random.seed(None)
    chosen = random.sample(keys, min(n_replacements, len(keys)))
    for key in chosen:
        syn = random.choice(SYNONYMS[key])
        new_text = new_text.replace(key, syn, 1)
    return new_text

def random_insert_legal_context(text):
    prefixes = [
        'Theo quy định của hợp đồng, ',
        'Căn cứ theo thỏa thuận giữa hai bên, ',
        'Theo các điều khoản đã ký kết, ',
        'Trong phạm vi hợp đồng này, ',
    ]
    suffixes = [
        ', trừ trường hợp bất khả kháng.',
        ', theo đúng quy định pháp luật hiện hành.',
        '.',
        ', phù hợp với các điều khoản đã thỏa thuận.',
    ]
    new_text = text.rstrip('.')
    if random.random() < 0.4:
        new_text = random.choice(prefixes) + new_text[0].lower() + new_text[1:]
    if random.random() < 0.3:
        new_text = new_text + random.choice(suffixes)
    else:
        new_text = new_text + '.'
    return new_text

random.seed(42)
aug_texts, aug_labels = list(texts), list(labels)

max_count = max(counts.values())
target_per_class = max_count

for lbl in INTENT_LABELS:
    cls_texts = [t for t, l in zip(texts, labels) if l == lbl]
    needed = target_per_class - counts[lbl]

    for _ in range(needed):
        base = random.choice(cls_texts)
        r = random.random()
        if r < 0.5:
            aug = synonym_replace(base, n_replacements=random.randint(1, 2))
        elif r < 0.8:
            aug = random_insert_legal_context(base)
        else:
            aug = random_insert_legal_context(synonym_replace(base))
        aug_texts.append(aug)
        aug_labels.append(lbl)

print(f'Sau augmentation: {len(aug_texts)} samples')
aug_counts = Counter(aug_labels)
for lbl in INTENT_LABELS:
    print(f'  {lbl:25}: {aug_counts[lbl]}')

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold

MODEL_NAME = 'vinai/phobert-base'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

label_ids = [LABEL2ID[l] for l in aug_labels]

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, val_idx = next(sss.split(aug_texts, label_ids))

train_texts  = [aug_texts[i]  for i in train_idx]
train_labels = [label_ids[i]  for i in train_idx]
val_texts    = [aug_texts[i]  for i in val_idx]
val_labels   = [label_ids[i]  for i in val_idx]

print(f'Train: {len(train_texts)} | Val: {len(val_texts)}')
print(f'Train distribution: {Counter([aug_labels[i] for i in train_idx])}')
print(f'Val   distribution: {Counter([aug_labels[i] for i in val_idx])}')

def tokenize_fn(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=256,
        padding=False,
    )

train_ds = Dataset.from_dict({'text': train_texts, 'labels': train_labels})
val_ds   = Dataset.from_dict({'text': val_texts,   'labels': val_labels})
train_ds = train_ds.map(tokenize_fn, batched=True).remove_columns(['text'])
val_ds   = val_ds.map(tokenize_fn,   batched=True).remove_columns(['text'])

In [7]:
from transformers import (
    TrainingArguments, Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)
from sklearn.metrics import f1_score, accuracy_score
import torch.nn.functional as F

total  = len(aug_labels)
n_cls  = len(INTENT_LABELS)
aug_c  = Counter(aug_labels)
class_weights = [total / (n_cls * aug_c[l]) for l in INTENT_LABELS]
weight_tensor = torch.tensor(class_weights, dtype=torch.float)

weights_dict = dict(zip(INTENT_LABELS, [round(w, 2) for w in class_weights]))
print(f"Class weights: {weights_dict}")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=n_cls,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)

def compute_metrics(eval_pred):
    logits, label_ids = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy':    round(accuracy_score(label_ids, preds), 4),
        'f1_macro':    round(f1_score(label_ids, preds, average='macro',    zero_division=0), 4),
        'f1_weighted': round(f1_score(label_ids, preds, average='weighted', zero_division=0), 4),
    }

class FocalLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop('labels')
        outputs = model(**inputs)
        logits  = outputs.logits

        weights = weight_tensor.to(logits.device)
        ce_loss = F.cross_entropy(logits, labels, weight=weights, reduction='none')

        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** 2 * ce_loss).mean()

        smooth_loss = -F.log_softmax(logits, dim=-1).mean()
        loss = 0.95 * focal_loss + 0.05 * smooth_loss

        return (loss, outputs) if return_outputs else loss

device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

print(f"Model: {MODEL_NAME}")
print(f"GPU: {device_name}")

In [8]:
num_epochs = 30
batch_size = 8
warmup_ratio = 0.1

args = TrainingArguments(
    output_dir                  = '/content/intent_phobert_checkpoints',
    num_train_epochs            = num_epochs,
    per_device_train_batch_size = batch_size,
    per_device_eval_batch_size  = 16,
    learning_rate               = 2e-5,
    weight_decay                = 0.01,
    warmup_ratio                = warmup_ratio,
    lr_scheduler_type           = 'cosine',
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1_macro',
    greater_is_better           = True,
    save_total_limit             = 3,
    logging_steps               = 10,
    report_to                   = 'none',
    seed                        = 42,
    fp16                        = torch.cuda.is_available(),
)

trainer = FocalLossTrainer(
    model           = model,
    args            = args,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    processing_class= tokenizer,
    data_collator   = DataCollatorWithPadding(tokenizer),
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=5)],
)

trainer.train()

In [9]:
from sklearn.metrics import classification_report, accuracy_score, f1_score

model.eval()
all_preds = []

for text in texts:
    enc = tokenizer(text, truncation=True, max_length=256, return_tensors='pt')
    enc = {k: v.to(model.device) for k, v in enc.items()}

    with torch.no_grad():
        logits = model(**enc).logits
        pred = torch.argmax(logits, dim=-1).item()

    all_preds.append(INTENT_LABELS[pred])

print('Classification Report:')
print(classification_report(labels, all_preds, target_names=INTENT_LABELS, zero_division=0))

acc = accuracy_score(labels, all_preds)
f1m = f1_score(labels, all_preds, average='macro', zero_division=0)
f1w = f1_score(labels, all_preds, average='weighted', zero_division=0)

print(f'Accuracy   : {acc:.4f}')
print(f'F1 macro   : {f1m:.4f}')
print(f'F1 weighted: {f1w:.4f}')

errors = [(t, p, g) for t, p, g in zip(texts, all_preds, labels) if p != g]

if errors:
    print(f'\nErrors ({len(errors)}/{len(texts)}):')
    for t, p, g in errors:
        print(f'  Pred={p:25} Gold={g:25}')
        print(f'  "{t[:100]}..."')
else:
    print('\nNo errors.')

In [10]:
from sklearn.model_selection import StratifiedKFold

orig_label_ids = [LABEL2ID[l] for l in labels]
n_folds = min(5, min(Counter(labels).values()))
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

fold_results = []
for fold, (tr_idx, te_idx) in enumerate(skf.split(texts, orig_label_ids)):
    fold_texts = [texts[i] for i in te_idx]
    fold_labels = [labels[i] for i in te_idx]

    fold_preds = []
    for text in fold_texts:
        enc = tokenizer(text, truncation=True, max_length=256, return_tensors='pt')
        enc = {k: v.to(model.device) for k, v in enc.items()}
        with torch.no_grad():
            logits = model(**enc).logits
            pred = torch.argmax(logits, dim=-1).item()
        fold_preds.append(INTENT_LABELS[pred])

    f_acc = accuracy_score(fold_labels, fold_preds)
    f_f1  = f1_score(fold_labels, fold_preds, average='macro', zero_division=0)
    fold_results.append({'accuracy': f_acc, 'f1_macro': f_f1})
    print(f'  Fold {fold+1}: Acc={f_acc:.3f}  F1={f_f1:.3f}')

avg_acc = np.mean([r['accuracy'] for r in fold_results])
avg_f1  = np.mean([r['f1_macro'] for r in fold_results])
std_acc = np.std([r['accuracy'] for r in fold_results])
std_f1  = np.std([r['f1_macro'] for r in fold_results])
print(f'\
  Average: Acc={avg_acc:.3f}+/-{std_acc:.3f}  F1={avg_f1:.3f}+/-{std_f1:.3f}')

In [12]:
import os
import json
import shutil

model.cpu()
model.eval()

if os.path.exists(MODEL_SAVE):
    shutil.rmtree(MODEL_SAVE)
os.makedirs(MODEL_SAVE, exist_ok=True)

model.save_pretrained(
    MODEL_SAVE,
    safe_serialization=True,
    max_shard_size='500MB',
)

tokenizer.save_pretrained(MODEL_SAVE)

config_path = os.path.join(MODEL_SAVE, 'label_config.json')
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump({
        'label2id': LABEL2ID,
        'id2label': {str(k): v for k, v in ID2LABEL.items()},
        'intent_labels': INTENT_LABELS,
    }, f, ensure_ascii=False, indent=2)

args_file = os.path.join(MODEL_SAVE, 'training_args.bin')
if os.path.exists(args_file):
    os.remove(args_file)

print(f"Model saved -> {MODEL_SAVE}")
total_size = 0

for fname in sorted(os.listdir(MODEL_SAVE)):
    fpath = os.path.join(MODEL_SAVE, fname)
    if os.path.isfile(fpath):
        fsize = os.path.getsize(fpath)
        total_size += fsize
        print(f"  {fname:<40} {fsize / (1024 * 1024):.1f} MB")

print(f"  {'TOTAL':<40} {total_size / (1024 * 1024):.1f} MB")

In [13]:
import shutil

zip_path = shutil.make_archive('/content/intent_phobert', 'zip', MODEL_SAVE)
print(f'Created: {zip_path} ({os.path.getsize(zip_path)/1024**2:.1f} MB)')

from google.colab import files
files.download(zip_path)